# Atividade Inteligencia Artificial

Você recebeu um arquivo contendo o histórico completo dos sorteios da Lotofácil, com os 15 números sorteados em cada concurso. Com base nesses dados históricos, realize uma análise estatística e, utilizando técnicas de previsão, machine learning ou análise de padrões, determine uma possível combinação de 15 números para o próximo sorteio.

**Metodologia utilizada:**

Para determinar a combinação de 15 números utilizei um modelo de score composto baseado em três critérios estatísticos extraídos do histórico de sorteios:

- Frequência histórica (peso 40%): contei quantas vezes cada número apareceu em todos os sorteios disponíveis. Números com maior frequência acumulada recebem maior peso no score.
- Atraso (peso 30%): calculei há quantos sorteios consecutivos cada número não é sorteado. Números com maior atraso acumulam uma pressão estatística de retorno.
- Tendência recente (peso 30%): analisei a frequência de cada número nos últimos 30 sorteios para capturar padrões de curto prazo.

Cada critério foi normalizado entre 0 e 1 antes de ser combinado. Os 15 números com maior score final formam a combinação sugerida.

Além disso, utilizei uma Arvore de Decisao para classificar cada número como quente (maior chance) ou frio (menor chance), usando como atributos a frequência histórica, o atraso e a frequência recente.

In [ ]:
# Instalar bibliotecas necessarias
!pip install scikit-learn pandas matplotlib seaborn --quiet

# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print('Bibliotecas importadas com sucesso!')

In [ ]:
# Carregar o arquivo CSV com o historico de sorteios
# Faca upload do arquivo no Colab antes de rodar esta celula

df = pd.read_csv('loto_facil_asloterias_todos_sorteios_2025_crescente.csv', sep=';')

# Identificar colunas das bolas
ball_cols = [c for c in df.columns if 'bola' in c]

print(f'Total de sorteios carregados: {len(df)}')
print(f'Concursos de {df["Concurso"].min()} a {df["Concurso"].max()}')
print()
df.head()

In [ ]:
# Analise de frequencia: contar quantas vezes cada numero apareceu

all_numbers = []
for _, row in df.iterrows():
    for c in ball_cols:
        all_numbers.append(int(row[c]))

freq = Counter(all_numbers)

freq_df = pd.DataFrame({
    'numero': list(range(1, 26)),
    'frequencia': [freq[n] for n in range(1, 26)],
    'percentual': [round(freq[n] / len(df) * 100, 1) for n in range(1, 26)]
})

print('Frequencia de cada numero:')
print(freq_df.to_string(index=False))

In [ ]:
# Grafico de frequencia historica

plt.figure(figsize=(14, 5))
media = freq_df['frequencia'].mean()
cores = ['#2196F3' if freq[n] >= media else '#90CAF9' for n in range(1, 26)]

plt.bar([str(n).zfill(2) for n in range(1, 26)],
        [freq[n] for n in range(1, 26)],
        color=cores, edgecolor='white', linewidth=0.5)

plt.axhline(media, color='red', linestyle='--', linewidth=1.5, label=f'Media = {media:.0f}')
plt.title('Frequencia Historica de Cada Numero - Lotofacil', fontsize=14, pad=15)
plt.xlabel('Numero')
plt.ylabel('Quantidade de sorteios')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Numero mais sorteado: {freq_df.loc[freq_df["frequencia"].idxmax(), "numero"]} ({freq_df["frequencia"].max()} vezes)')
print(f'Numero menos sorteado: {freq_df.loc[freq_df["frequencia"].idxmin(), "numero"]} ({freq_df["frequencia"].min()} vezes)')

In [ ]:
# Analise de atraso: quantos sorteios cada numero ficou sem aparecer
# O dataframe esta em ordem decrescente: indice 0 = sorteio mais recente

delay = {}
for n in range(1, 26):
    for idx, row in df.iterrows():
        nums = set(int(row[c]) for c in ball_cols)
        if n in nums:
            delay[n] = idx
            break
    else:
        delay[n] = len(df)

delay_df = pd.DataFrame({
    'numero': list(range(1, 26)),
    'atraso': [delay[n] for n in range(1, 26)]
})

print('Atraso de cada numero (sorteios desde a ultima aparicao):')
print(delay_df.to_string(index=False))
print(f'\nMaior atraso: numero {delay_df.loc[delay_df["atraso"].idxmax(), "numero"]} ({delay_df["atraso"].max()} sorteios)')

In [ ]:
# Grafico de atraso por numero

plt.figure(figsize=(14, 5))
cores_atraso = ['#F44336' if delay[n] >= 2 else '#FF9800' if delay[n] == 1 else '#4CAF50'
                for n in range(1, 26)]

plt.bar([str(n).zfill(2) for n in range(1, 26)],
        [delay[n] for n in range(1, 26)],
        color=cores_atraso, edgecolor='white')

p1 = mpatches.Patch(color='#4CAF50', label='Atraso 0 (apareceu no ultimo sorteio)')
p2 = mpatches.Patch(color='#FF9800', label='Atraso 1')
p3 = mpatches.Patch(color='#F44336', label='Atraso >= 2')
plt.legend(handles=[p1, p2, p3], fontsize=9)

plt.title('Atraso de Cada Numero - Sorteios Consecutivos sem Aparecer', fontsize=13, pad=15)
plt.xlabel('Numero')
plt.ylabel('Atraso (sorteios)')
plt.tight_layout()
plt.show()

In [ ]:
# Tendencia recente: frequencia nos ultimos 30 sorteios

recent = df.head(30)
recent_counts = Counter()
for _, row in recent.iterrows():
    for c in ball_cols:
        recent_counts[int(row[c])] += 1

# Mapa de calor 5x5
matriz = np.array([recent_counts.get(n, 0) for n in range(1, 26)]).reshape(5, 5)
labels_mat = np.array([str(n).zfill(2) for n in range(1, 26)]).reshape(5, 5)

plt.figure(figsize=(8, 6))
ax = sns.heatmap(matriz, annot=labels_mat, fmt='', cmap='YlOrRd',
                 linewidths=2, linecolor='white',
                 cbar_kws={'label': 'Aparicoes nos ultimos 30 sorteios'},
                 annot_kws={'size': 14, 'weight': 'bold'})

for i in range(5):
    for j in range(5):
        n = i * 5 + j + 1
        ax.text(j + 0.5, i + 0.75, f'({recent_counts.get(n,0)}x)',
                ha='center', va='center', fontsize=9, color='#555')

plt.title('Mapa de Calor - Frequencia nos Ultimos 30 Sorteios', fontsize=13, pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Calcular o score composto para cada numero
# Frequencia historica (40%) + Atraso (30%) + Tendencia recente (30%)

max_freq  = max(freq.values())
max_delay = max(delay.values())
max_rec   = max(recent_counts.values())

scores = {}
for n in range(1, 26):
    norm_freq  = freq[n] / max_freq
    norm_delay = delay[n] / max_delay
    norm_rec   = recent_counts.get(n, 0) / max_rec
    scores[n]  = 0.40 * norm_freq + 0.30 * norm_delay + 0.30 * norm_rec

score_df = pd.DataFrame({
    'numero':      list(range(1, 26)),
    'freq_hist':   [freq[n] for n in range(1, 26)],
    'atraso':      [delay[n] for n in range(1, 26)],
    'freq_rec30':  [recent_counts.get(n, 0) for n in range(1, 26)],
    'score_final': [round(scores[n], 4) for n in range(1, 26)]
}).sort_values('score_final', ascending=False)

print('Ranking por Score Composto:')
print(score_df.to_string(index=False))

In [ ]:
# Selecionar os 15 numeros com maior score

top15 = sorted(score_df.head(15)['numero'].tolist())

print('Combinacao sugerida (15 numeros com maior score):')
print()
print('  ' + '  '.join([str(n).zfill(2) for n in top15]))
print()

# Grafico do ranking
plt.figure(figsize=(14, 5))
nums_ord   = score_df['numero'].tolist()
scores_ord = score_df['score_final'].tolist()
cores_sc   = ['#1976D2' if n in top15 else '#B0BEC5' for n in nums_ord]

plt.bar([str(n).zfill(2) for n in nums_ord], scores_ord, color=cores_sc, edgecolor='white')
plt.axhline(scores_ord[14], color='red', linestyle='--', linewidth=1.5,
            label=f'Corte top-15 (score = {scores_ord[14]:.4f})')

p1 = mpatches.Patch(color='#1976D2', label='Top 15 selecionados')
p2 = mpatches.Patch(color='#B0BEC5', label='Nao selecionados')
plt.legend(handles=[p1, p2])

plt.title('Score Composto por Numero - Modelo de IA Ponderado', fontsize=13, pad=15)
plt.xlabel('Numero (ordenado por score)')
plt.ylabel('Score final (0 a 1)')
plt.tight_layout()
plt.show()

In [ ]:
# Arvore de Decisao: classificar numeros como quente ou frio

# Criar dataset com os atributos de cada numero
ml_df = pd.DataFrame({
    'numero':     list(range(1, 26)),
    'freq_hist':  [freq[n] for n in range(1, 26)],
    'atraso':     [delay[n] for n in range(1, 26)],
    'freq_rec30': [recent_counts.get(n, 0) for n in range(1, 26)],
    'score':      [scores[n] for n in range(1, 26)]
})

# Rotulo: quente se score >= mediana, frio caso contrario
mediana_score = ml_df['score'].median()
ml_df['classe'] = ml_df['score'].apply(lambda x: 'quente' if x >= mediana_score else 'frio')

# Dividir em treino (70%) e teste (30%)
X = ml_df[['freq_hist', 'atraso', 'freq_rec30']]
y = ml_df['classe']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Treinar o modelo
modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

# Avaliar o modelo
y_pred = modelo.predict(X_test)
print(f'Acuracia do modelo: {accuracy_score(y_test, y_pred):.2%}')
print()
print('Relatorio de Classificacao:')
print(classification_report(y_test, y_pred))

In [ ]:
# Visualizar a Arvore de Decisao

plt.figure(figsize=(14, 7))
plot_tree(
    modelo,
    feature_names=['freq_hist', 'atraso', 'freq_rec30'],
    class_names=modelo.classes_,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Arvore de Decisao - Classificacao de Numeros Quentes e Frios', fontsize=13, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Classificar todos os 25 numeros com o modelo treinado

ml_df['predicao'] = modelo.predict(X)
quentes_ml = sorted(ml_df[ml_df['predicao'] == 'quente']['numero'].tolist())

print('Classificacao de todos os numeros pela Arvore de Decisao:')
print()
print(ml_df[['numero', 'freq_hist', 'atraso', 'freq_rec30', 'score', 'classe', 'predicao']].to_string(index=False))
print()
print(f'Numeros classificados como quente pelo modelo: {quentes_ml}')

In [ ]:
# Combinacao final e justificativa

top15 = sorted(score_df.head(15)['numero'].tolist())

print('=' * 50)
print('COMBINACAO SUGERIDA PARA O PROXIMO SORTEIO')
print('=' * 50)
print()
print('  ' + '  '.join([str(n).zfill(2) for n in top15]))
print()
print('Justificativa:')
print('  Os numeros 19 e 24 lideram o ranking pois nao apareceram nos ultimos')
print('  3 sorteios consecutivos, que e o maior atraso observado na base,')
print('  e ainda possuem alta frequencia historica (176 e 183 vezes).')
print('  O numero 11 e o mais sorteado de toda a base (190 vezes) e esta com')
print('  2 sorteios de atraso. Os demais numeros foram escolhidos por equilibrar')
print('  bem os tres criterios: frequencia acima da media, atraso positivo e')
print('  boa presenca nos ultimos 30 sorteios.')
print()
print('Aviso: este e um exercicio educacional. A Lotofacil e aleatoria')
print('e nenhum metodo estatistico garante acerto.')

In [ ]:
# Visualizar a combinacao final como volante grafico

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_facecolor('#f5f5f5')
fig.patch.set_facecolor('#f5f5f5')

for i, n in enumerate(range(1, 26)):
    row = i // 5
    col = i % 5
    x = col + 0.5
    y = 4.5 - row
    selecionado = n in top15
    cor = '#1565C0' if selecionado else '#E0E0E0'
    cor_texto = 'white' if selecionado else '#757575'
    circle = plt.Circle((x, y), 0.38, color=cor, zorder=2)
    ax.add_patch(circle)
    ax.text(x, y, str(n).zfill(2), ha='center', va='center',
            fontsize=14, fontweight='bold', color=cor_texto, zorder=3)

ax.set_title('Volante - Numeros Sugeridos pelo Modelo de IA',
             fontsize=13, pad=20, fontweight='bold')

p1 = mpatches.Patch(color='#1565C0', label='Selecionado')
p2 = mpatches.Patch(color='#E0E0E0', label='Nao selecionado')
ax.legend(handles=[p1, p2], loc='lower center',
          bbox_to_anchor=(0.5, -0.05), ncol=2, frameon=False)

plt.tight_layout()
plt.show()